In [ ]:
!pip install groq
!sudo apt-get update
!sudo apt-get install -y tesseract-ocr
!tesseract --version
!pip install pytesseract pdf2image
!pip install -U langchain-community
!pip install pinecone
!pip install gradio
!apt-get install poppler-utils
!pip install tavily-python

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:4 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 

In [ ]:
import os
import gradio as gr
from langchain_core.prompts import PromptTemplate
from tavily import TavilyClient
from google.colab import files
from pinecone import Pinecone, ServerlessSpec

from groq import Groq
from sentence_transformers import SentenceTransformer
from pdf2image import convert_from_path
import pytesseract
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
model = SentenceTransformer('sentence-transformers/all-roberta-large-v1')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def extract_text_from_pdf(pdf_path):
    print("Using Tesseract for OCR on scanned PDF...")
    images = convert_from_path(pdf_path)
    print("converted to images")
    text = ""
    custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
    for i, image in enumerate(images):
        text += pytesseract.image_to_string(image, config=custom_config) + "\n"

    return text

pdf_path = "/content/drive/MyDrive/CamScanner 08-21-2024 22.31.pdf"
extracted_text = extract_text_from_pdf(pdf_path)
print("Text extracted")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
chunks = text_splitter.create_documents([extracted_text])
print("Documents Created")
def generate_embeddings(texts):
    return model.encode([text.page_content for text in texts])

embeddings = generate_embeddings(chunks)
print("Embeddings generated")



Using Tesseract for OCR on scanned PDF...
converted to images
Text extracted
Documents Created
Embeddings generated


In [ ]:
pc = Pinecone(api_key="paste your pinecone api here")
index = pc.Index(name="index name here", host="host link here")

In [ ]:
vectors_to_insert = []
for i, embedding in enumerate(embeddings):
    vectors_to_insert.append((
        str(i),
        embedding.tolist(),
        {"chunk_id": i, "content": chunks[i].page_content[:100]}
    ))
for i in range(0,len(vectors_to_insert),400):
    index.upsert(vectors=vectors_to_insert[i:i+400])

In [ ]:
client = Groq(api_key="groq api key here")
tavily_client = TavilyClient(api_key="tavily api key here")

def generate_embeddings(texts):
    return model.encode(texts)

def get_response_from_lama(prompt: str, model: str = "openai/gpt-oss-20b") -> str:
    try:
        chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model=model,
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        raise ValueError(f"Groq API call failed: {e}")

def get_answer_from_query(user_query):
  template = """Analyze the following Question and give one word answer of YES if the question is about pakistan history.
   answer with NO otherwise. Question: {query}"""

  prompt = PromptTemplate(input_variables=["query"],template=template)
  final_prompt = prompt.format(query=user_query)
  response = get_response_from_lama(final_prompt)
  if response == "NO":
    return ["Query was not About Pakistan History","Error"]

  query_embedding = generate_embeddings([user_query])[0]
  query_result = index.query(vector=query_embedding.tolist(), top_k=3)
  reply_from = ''
  if(query_result['matches'][0]['score']>0.65):
    context = "\n".join([" ".join(match['values']) for match in query_result['matches']])
    reply_from = 'RAG'
  else:
    reply = tavily_client.search(user_query)
    relevant_content = [i["content"] for i in reply["results"] if i.get("score",0) > 0.65]
    context = "\n\n".join(relevant_content)
    reply_from = 'Web'

  llama_prompt = f"Context:\n{context}\n\nUser Query: {user_query}\nAnswer:"
  response = get_response_from_lama(llama_prompt)
  return response,reply_from

In [ ]:
demo = gr.Interface(
    fn=get_answer_from_query,
    inputs=[gr.Textbox(label="Prompt")],
    outputs=[gr.Textbox(lines=5,label="Result"),gr.Textbox(label="Type")],
    title="Chatbot"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://579f9c9ff63c02e36d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
